# OCTRON YOLO prediction — advanced Python workflow

This notebook uses the lower-level `YOLO_octron.predict_batch()` generator directly.

Use this workflow when you need:

- custom progress handling,
- direct access to progress dictionaries,
- custom tracker parameter overrides,
- custom tracker YAML files,
- explicit region-property lists,
- custom `extra_properties` callables for `skimage.measure.regionprops_table`.

For the simpler CLI-like Python wrapper, see `OCTRON_YOLO_predict_simple.ipynb`.

Results loading is intentionally not covered here; see `OCTRON_results_loading.ipynb`.


## 1. Imports and object setup


In [ ]:
from pathlib import Path
import numpy as np

import octron
from octron.yolo_octron.yolo_octron import YOLO_octron
from octron.yolo_octron.constants import DEFAULT_REGION_PROPERTIES, ALL_REGION_PROPERTIES
from octron.tracking.helpers.tracker_checks import load_boxmot_trackers, list_available_trackers

octron_base_path = Path(octron.__file__).resolve().parent

yolo = YOLO_octron(
    models_yaml_path=None,   # None = use the packaged yolo_models.yaml
    project_path=None,       # Optional OCTRON project directory
    clean_training_dir=False,
)


## 2. List available trackers

Tracker IDs are what you pass as `tracker_name`. Display names are accepted too, but using the IDs is usually easiest for scripts.


In [ ]:
trackers_yaml = octron_base_path / "tracking" / "boxmot_trackers.yaml"
trackers_dict = load_boxmot_trackers(trackers_yaml)
available = list_available_trackers(trackers_dict)

for tracker_id, display_name in available.items():
    print(f"{tracker_id:15s}  ({display_name})")


## 3. Configure paths and common parameters


In [ ]:
video_paths = [
    Path("path/to/video_1.mp4"),
    Path("path/to/video_2.mp4"),
]

# predict_batch expects a resolved .pt model path.
model_path = Path("path/to/model/training/weights/best.pt")

# Final output root. None writes octron_predictions/ next to each video.
output_dir = None
# output_dir = Path("path/to/output_root")

# Optional fast local staging directory; useful for network/external final output locations.
local_cache_dir = None
# local_cache_dir = Path("/fast/local/cache")

device = "auto"  # or: "cuda", "mps", "cpu"
tracker_name = "hybridsort"

conf_thresh = 0.5
iou_thresh = 0.7
skip_frames = 0
overwrite = False


## 4. Region properties (`region_properties`)

`region_properties` controls scalar measurements extracted from each segmentation mask.

- `None`: bbox-only mode (fastest)
- `DEFAULT_REGION_PROPERTIES`: standard minimal detailed output
- explicit tuple/list: your own selection
- all flattened values from `ALL_REGION_PROPERTIES`: everything listed in OCTRON

Detection models do not produce masks, so region properties are ignored for detection-only models.


In [ ]:
region_properties = None

# Standard minimal detailed output:
# region_properties = DEFAULT_REGION_PROPERTIES

# Explicit selection:
# region_properties = ("area", "eccentricity", "solidity")

# All available scalar region properties:
# region_properties = tuple(name for group in ALL_REGION_PROPERTIES.values() for name in group)

print("Available region properties:")
for group, names in ALL_REGION_PROPERTIES.items():
    print(f"{group}:")
    for name in names:
        print(f"  - {name}")


## 5. Custom measurement functions (`extra_properties`)

`extra_properties` is Python-only and is not exposed in the CLI. Each callable is passed to `skimage.measure.regionprops_table`; its `__name__` becomes the CSV column name.

Functions can either accept only `regionmask`, or accept both `regionmask` and `intensity_image`.


In [ ]:
def compactness(regionmask):
    """Example mask-only custom property."""
    area = np.sum(regionmask)
    if area == 0:
        return 0.0
    # Very simple edge estimate for demonstration purposes.
    vertical_edges = np.sum(regionmask ^ np.roll(regionmask, 1, axis=0))
    horizontal_edges = np.sum(regionmask ^ np.roll(regionmask, 1, axis=1))
    perimeter = vertical_edges + horizontal_edges
    return (4 * np.pi * area) / (perimeter ** 2) if perimeter > 0 else 0.0


def mean_brightness(regionmask, intensity_image):
    """Example intensity-based custom property."""
    return float(np.mean(intensity_image[regionmask])) if np.any(regionmask) else 0.0


extra_properties = None
# extra_properties = (compactness, mean_brightness)


## 6. Optional tracker customization

There are two levels of tracker customization.

### Option A: parameter overrides

`tracker_params` updates the `current_value` fields in the resolved tracker config.

### Option B: custom tracker YAML

Create a starting YAML file with:

```bash
octron dump-tracker-config hybridsort -o my_hybridsort.yaml
```

Edit the YAML, then pass it as `tracker_cfg_path`. When `tracker_cfg_path` is set, it overrides `tracker_name`.


In [ ]:
# Option A: small overrides on top of a packaged tracker config.
tracker_params = None
# tracker_params = {
#     "max_age": 100,
#     "min_hits": 3,
# }

# Option B: fully custom tracker config YAML.
tracker_cfg_path = None
# tracker_cfg_path = Path("my_hybridsort.yaml")

if tracker_cfg_path is not None:
    tracker_name_for_run = None
else:
    tracker_name_for_run = tracker_name


## 7. Run `predict_batch()` with custom progress handling

`predict_batch()` is a generator. Each yielded dictionary has a `stage` field. The most common stages are:

- `processing`: a frame was processed
- `video_complete`: a video finished successfully
- `skipped_video`: output already existed and `overwrite=False`
- `complete`: the whole batch finished


In [ ]:
for progress in yolo.predict_batch(
    videos=video_paths,
    model_path=model_path,
    device=device,
    tracker_name=tracker_name_for_run,
    tracker_cfg_path=tracker_cfg_path,
    tracker_params=tracker_params,
    skip_frames=skip_frames,
    one_object_per_label=False,
    region_properties=region_properties,
    extra_properties=extra_properties,
    iou_thresh=iou_thresh,
    conf_thresh=conf_thresh,
    opening_radius=0,
    overwrite=overwrite,
    buffer_size=500,
    output_dir=output_dir,
    local_cache_dir=local_cache_dir,
):
    stage = progress.get("stage")

    if stage == "processing":
        # `frame` counts processed frames in the analysis iterator, not necessarily original video frame index.
        if progress.get("frame", 0) % 100 == 0:
            print(
                f"Video {progress['video_index'] + 1}/{progress['total_videos']} "
                f"({progress['video_name']}): "
                f"{progress['frame']}/{progress['total_frames']}"
            )

    elif stage == "video_complete":
        print(f"✓ Completed {progress['video_name']}")
        print(f"  Results saved to: {progress['save_dir']}")

    elif stage == "skipped_video":
        print(f"Skipped {progress['video_name']} (already exists). Use overwrite=True to replace it.")

    elif stage == "complete":
        print("✓ All videos processed")


## 8. When to prefer the simple wrapper

If you do not need custom progress handling or `extra_properties`, use `run_predict()` instead. It is the same high-level path used by the CLI and handles common conveniences such as directory expansion, `device="auto"`, model directory resolution, and progress printing.

See `OCTRON_YOLO_predict_simple.ipynb`.
